# Protocol 7 — Disorders of Consciousness Biomarker

Simulates Protocol 7 (*Disorders of Consciousness Biomarker: Joint Roles of Ignition Capacity and
Interoceptive Precision*), updated to **protocol specification v1.1.0**. The fixed cohort has 80 patients
(VS/UWS 30, MCS 30, EMCS 20) and 30 healthy controls across 30 sites.

* **Pred 7.G** *(primary confirmatory)* — Baseline HEP alone predicts 3-month **CRS-R** total in patients beyond
  age and etiology: LRT for HEP in `CRS-R_3mo ~ HEP + age + etiology + (1 | site)`, α = 0.048, positive
  coefficient. Marginal ΔR² (Nakagawa & Schielzeth) is reported against the SESOI of 0.05. *Falsified*: 90 % CI
  for ΔR² wholly below 0.05 **and** BF₀₁ ≥ 6. Controls are excluded.
* **Pred 7.A** *(estimation-first)* — Joint HEP + PCI improves cross-validated McFadden R² for **baseline**
  consciousness level by ΔR² ≥ 0.05 (95 % CI lower bound > 0) over the best univariate model, in **both** the
  four-group and patient-only populations.
* **Pred 7.B** *(secondary)* — HEP: MCS > VS/UWS (d > 0.5), monotonic four-group gradient, and the DMN double
  dissociation (DMN–PCI r > 0.50, DMN–HEP r < 0.20).
* **Pred 7.C** *(secondary, conditional)* — CCRC (5 % CO₂ / 95 % O₂, ≤ 90 s) raises PCI ≥ 10 % in MCS but not
  VS/UWS, and beats **both** the auditory and the carrier-matched sham controls by ≥ 5 % of baseline PCI
  (one-sided α = 0.025 each). ΔPCI correlates with ΔHEP (r > 0.4).
* **Pred 7.D** *(secondary)* — Between-participant Spearman r(baseline HEP, CRS-R) > 0.40 at 3 and 6 months
  (6 months is the primary prognostic endpoint). The joint model adds ΔR² ≥ 0.05 at 6 months.
* **Pred 7.E / 7.F** *(exploratory, no framework weight)* — Infraslow–theta PAC; body-focus self-report in
  communicating EMCS/MCS+ patients only.

> All data are synthetic. CCRC requires physician-approved delivery; the carrier-matched sham awaits
> composition and authorization. LOCF is not permitted.
>
> **EP-7 | Paper 3 | Clinical | Depends on: EP-0 (HEP proxy gate) | Spec v1.1.0, pre-collection |
> N = 110 fixed (80 patients + 30 controls); Pred 7.G analysable floor = 60**

### What changed from v1.0

| Item | v1.0 notebook | v1.1 specification |
|---|---|---|
| Primary test | 7.A joint ΔR² on 3-month CRS-R | **7.G**: patient-only HEP LRT, α = 0.048, site random intercept (power 0.68 at N = 80) |
| Pred 7.A | 3-month outcome, ΔR² ≥ 0.05 | *Baseline* consciousness level, cross-validated McFadden ΔR², estimation-first, four-group **and** patient-only |
| Pred 7.C | white-noise control | auditory **and** carrier-matched sham controls, ≥ 5 % margins, one-sided α = 0.025 each |
| Pred 7.D | within-participant r | **between**-participant Spearman r at 3 and 6 months; 6-month ΔR² |
| Pred 7.E | somatic-bias self-report | infraslow–theta PAC (exploratory); self-report moved to 7.F |
| Missing data | LOCF excluded | LOCF not permitted; MICE (m = 50) sensitivity; joint trajectory–survival for 7.D |

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit
from scipy.stats import chi2, ncf, pearsonr, spearmanr, ttest_ind, ttest_rel
from scipy.stats import f as f_dist

from apgi.core import BETA_SM_DEFAULT, compute_pi_i_eff

In [ ]:
# Load the v1.1 protocol specification so every threshold below is read from
# the registered JSON rather than retyped by hand.
def load_protocol(filename):
    for root in (Path.cwd(), *Path.cwd().parents):
        path = root / "protocols" / filename
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    raise FileNotFoundError(f"protocols/{filename} not found above {Path.cwd()}")


PROTO = load_protocol("protocol_7_doc_biomarker.json")
assert PROTO["version"] == "1.1.0", PROTO["version"]
PRED = {p["id"]: p for p in PROTO["sub_predictions"]}
print(f"{PROTO['protocol_id']}  spec v{PROTO['version']}  status={PROTO['study_status']}")
for pid, p in PRED.items():
    print(f"  {pid:<9} [{p['status']}]  {p['claim'][:78]}")


def verdict(confirmed, falsified=False):
    """v1.1 three-way decision: failing to confirm is not falsification."""
    if confirmed:
        return "CONFIRMED (synthetic)"
    if falsified:
        return "FALSIFIED (synthetic)"
    return "INDETERMINATE (not confirmed; falsification rule not met)"

In [ ]:
def jzs_bf10(t, n, r=0.707):
    """JZS Bayes factor for a one-sample / paired t-test (Rouder et al., 2009).

    Cauchy prior scale r = 0.707, as registered for the v1.1 Bayesian tests.
    The likelihood ratio is formed inside the integral to avoid underflow.
    """
    nu = n - 1

    def integrand(g):
        ratio = ((1 + t**2 / ((1 + n * g * r**2) * nu)) / (1 + t**2 / nu)) ** (-(nu + 1) / 2)
        prior = (2 * np.pi) ** -0.5 * g**-1.5 * np.exp(-1 / (2 * g))
        return (1 + n * g * r**2) ** -0.5 * ratio * prior

    return float(integrate.quad(integrand, 0, np.inf, limit=200)[0])

## 1 — Protocol 7 cohort (v1.1)

In [ ]:
P, POWER = PROTO["participants"], PROTO["power_analysis"]
GROUPS = ["VS/UWS", "MCS", "EMCS", "Controls"]
N_PER_GROUP = dict(zip(GROUPS, [P["groups"][k] for k in ["VS_UWS", "MCS", "EMCS", "healthy_controls"]]))
N_SITES = P["sites"]
THR_G = PRED["Pred 7.G"]
ALPHA_G = THR_G["confirming_thresholds"]["alpha"]  # 0.048
SESOI = THR_G["falsifying_thresholds"]["delta_r_squared_ci_upper_strict_maximum"]  # 0.05
assert sum(N_PER_GROUP.values()) == P["n_target"] and N_PER_GROUP["Controls"] == P["n_target"] - P["patient_n_target"]

# Illustrative generative values (synthetic; not registered)
PI_I_BY_GROUP = {"VS/UWS": 0.3, "MCS": 0.8, "EMCS": 1.1, "Controls": 1.2}
PCI_BY_GROUP = {"VS/UWS": 0.20, "MCS": 0.36, "EMCS": 0.44, "Controls": 0.52}
CRS_R_BASELINE = {"VS/UWS": 5.0, "MCS": 10.0, "EMCS": 18.0, "Controls": 23.0}

print(f"N = {P['n_target']} ({P['design']}): {N_PER_GROUP}; {N_SITES} sites")
print(f"Pred 7.G α = {ALPHA_G}, SESOI ΔR² = {SESOI}, registered power {POWER['power']} at N = {POWER['patient_n_target']}")
rng = np.random.default_rng(7)

rows = []
for g in GROUPS:
    for _ in range(N_PER_GROUP[g]):
        pi_eff = compute_pi_i_eff(PI_I_BY_GROUP[g], BETA_SM_DEFAULT, float(rng.uniform(0, 0.5)))
        rows.append(dict(group=g, site=int(rng.integers(N_SITES)), age=rng.normal(45, 15),
                         tbi=int(rng.random() < 0.5), hep=pi_eff + rng.normal(0, 0.60),
                         pci=rng.normal(PCI_BY_GROUP[g], 0.06), dmn=0.0))
hep = np.array([r["hep"] for r in rows])
pci = np.array([r["pci"] for r in rows])
group = np.array([r["group"] for r in rows])
site = np.array([r["site"] for r in rows])
age = np.array([r["age"] for r in rows])
tbi = np.array([r["tbi"] for r in rows])
patients = group != "Controls"
dmn = 0.8 * (pci - pci.mean()) / pci.std() + rng.normal(0, 0.55, len(pci))  # DMN connectivity tracks PCI

# 3- and 6-month CRS-R (patients); HEP contributes recovery beyond severity, age and etiology
site_effect = rng.normal(0, 1.0, N_SITES)
hep_z = (hep - hep[patients].mean()) / hep[patients].std()
crs3 = np.array([CRS_R_BASELINE[g] for g in group]) + 0.8 * hep_z - 0.04 * (age - 45) + 1.0 * tbi \
    + site_effect[site] + rng.normal(0, 3.0, len(group))
# Both markers carry prognostic information beyond group severity (the joint-model hypothesis)
pci_within = pci - np.array([pci[group == g].mean() for g in group])
hep_within = hep - np.array([hep[group == g].mean() for g in group])
crs6 = crs3 + 1.0 + 1.6 * hep_within / hep_within.std() + 2.2 * pci_within / pci_within.std()             + rng.normal(0, 1.5, len(group))
crs3, crs6 = np.clip(crs3, 0, 23), np.clip(crs6, 0, 23)
observed3 = patients & (rng.random(len(group)) > 0.08)  # ~8 % missing; LOCF not permitted
print(f"Patients with observed 3-month CRS-R: {observed3.sum()} (analysable floor {P['pred_7g_analyzable_floor']})")

## 2 — Pred 7.G (primary): HEP → 3-month CRS-R beyond age and etiology

Maximum-likelihood random-intercept model, fitted by profiling the site-variance ratio. Inference uses an LRT
on the HEP term, marginal ΔR², and a whole-site bootstrap (≥ 10 sites). The registered analysis uses
2000 bootstrap resamples; this demo uses fewer.

In [ ]:
def lmm_ml(y, X, groups):
    """ML fit of y = Xβ + b_site + ε; returns (loglik, β, σ²_b, σ²)."""
    uniq = np.unique(groups)
    Z = (groups[:, None] == uniq[None, :]).astype(float)
    n = len(y)

    def profile(log_lam):
        V = np.eye(n) + np.exp(log_lam) * Z @ Z.T
        Vi = np.linalg.inv(V)
        beta = np.linalg.solve(X.T @ Vi @ X, X.T @ Vi @ y)
        r = y - X @ beta
        s2 = r @ Vi @ r / n
        _, logdet = np.linalg.slogdet(V)
        return 0.5 * (n * np.log(2 * np.pi * s2) + logdet + n), beta, s2

    opt = minimize_scalar(lambda v: profile(v)[0], bounds=(-10, 4), method="bounded")
    nll, beta, s2 = profile(opt.x)
    return -nll, beta, np.exp(opt.x) * s2, s2


def marginal_r2(X, beta, s2b, s2):
    v_fixed = np.var(X @ beta)
    return v_fixed / (v_fixed + s2b + s2)


def fit_7g(idx):
    y = crs3[idx]
    X0 = np.column_stack([np.ones(len(idx)), age[idx], tbi[idx]])
    X1 = np.column_stack([X0, hep[idx]])
    ll0, b0, s2b0, s20 = lmm_ml(y, X0, site[idx])
    ll1, b1, s2b1, s21 = lmm_ml(y, X1, site[idx])
    return dict(lrt=2 * (ll1 - ll0), beta_hep=b1[-1],
                dr2=marginal_r2(X1, b1, s2b1, s21) - marginal_r2(X0, b0, s2b0, s20),
                se=None)


idx_g = np.flatnonzero(observed3)
res = fit_7g(idx_g)
p_lrt = chi2.sf(res["lrt"], 1)

# Whole-site bootstrap for ΔR² (≥ 10 sites present)
N_BOOT = 300
sites_present = np.unique(site[idx_g])
boot_rng = np.random.default_rng(13)
boot = []
for _ in range(N_BOOT):
    draw = boot_rng.choice(sites_present, len(sites_present))
    idx_b = np.concatenate([idx_g[site[idx_g] == s] for s in draw])
    boot.append(fit_7g(idx_b)["dr2"])
ci90 = np.percentile(boot, [5, 95])
ci95 = np.percentile(boot, [2.5, 97.5])

# Approximate BF for the standardized HEP slope (Cauchy 0.707) from the LRT's signed root
t_hep = np.sign(res["beta_hep"]) * np.sqrt(res["lrt"])
bf01 = 1 / jzs_bf10(t_hep, len(idx_g))

confirmed_7G = p_lrt < ALPHA_G and res["beta_hep"] > 0
falsified_7G = (ci90[1] < SESOI and bf01 >= THR_G["falsifying_thresholds"]["bf01_minimum"]) or \
    (p_lrt < ALPHA_G and res["beta_hep"] < 0)
print(f"N = {len(idx_g)} patients, {len(sites_present)} sites")
print(f"LRT χ²(1) = {res['lrt']:.2f}, p = {p_lrt:.1e}; β_HEP = {res['beta_hep']:+.2f} CRS-R points per HEP unit")
print(f"Marginal ΔR² = {res['dr2']:.3f}  (90% CI {ci90[0]:.3f}–{ci90[1]:.3f}; 95% CI {ci95[0]:.3f}–{ci95[1]:.3f}); "
      f"BF₀₁ ≈ {bf01:.2f}")
print(f"Pred 7.G → {verdict(confirmed_7G, falsified_7G)}")
if confirmed_7G and ci95[1] < SESOI:
    print("   reliable but below the ΔR² = 0.05 clinically meaningful floor — independent replication required")

### Registered power cross-check

The JSON gives f² = 0.0769 at α = 0.048 with power 0.68 at N = 80, plus a sensitivity table. The noncentral-F
calculation below (one tested predictor, four nuisance terms, λ = f²·N) reproduces those figures.

In [ ]:
f2 = POWER["effect_size_f_squared"]
for n_, reg in [(80, POWER["power"])] + [(r["n"], r["power"]) for r in POWER["power_sensitivity"]]:
    df2 = n_ - 5
    crit = f_dist.ppf(1 - ALPHA_G, 1, df2)
    pw = ncf.sf(crit, 1, df2, f2 * n_)
    print(f"  N = {n_:>3}: computed power {pw:.2f}   registered {reg:.2f}")
print(f"  N for 80% power (registered): {POWER['n_required_for_80_percent_power']}")

## 3 — Pred 7.A (estimation-first): joint HEP + PCI for baseline consciousness level

Proportional-odds logistic models with five-fold cross-validated McFadden R². The registered interval uses
10 000 BCa resamples; this demo uses a small percentile bootstrap. Etiology enters only the patient-only model.

In [ ]:
def ordinal_nll(params, X, y, K):
    beta, cuts = params[:X.shape[1]], np.cumsum(np.r_[params[X.shape[1]], np.exp(params[X.shape[1] + 1:])])
    eta = X @ beta
    cdf = expit(cuts[None, :] - eta[:, None])
    cdf = np.column_stack([np.zeros(len(y)), cdf, np.ones(len(y))])
    p = np.clip(cdf[np.arange(len(y)), y + 1] - cdf[np.arange(len(y)), y], 1e-12, 1)
    return -np.sum(np.log(p))


FOLD_OF = np.random.default_rng(0).permutation(len(group)) % 5  # folds keyed to participant ID


def cv_mcfadden(X, y, K, ids, folds=5):
    fold = FOLD_OF[ids]  # bootstrap duplicates of one participant always share a fold (no leakage)
    ll_model = ll_null = 0.0
    for k in range(folds):
        tr, te = fold != k, fold == k
        x0 = np.r_[np.zeros(X.shape[1]), -1.0, np.zeros(K - 2)]
        def ridge(p):  # small L2 penalty stabilizes small folds
            return ordinal_nll(p, X[tr], y[tr], K) + 0.05 * np.sum(p[:X.shape[1]] ** 2)

        fit = minimize(ridge, x0, method="BFGS")
        ll_model -= ordinal_nll(fit.x, X[te], y[te], K)
        props = np.bincount(y[tr], minlength=K) / tr.sum()
        ll_null += np.sum(np.log(np.clip(props[y[te]], 1e-12, 1)))
    return 1 - ll_model / ll_null


def delta_r2_7a(ids, extra=None):
    y = np.array([GROUPS.index(g) for g in group[ids]])
    K = len(np.unique(y))
    y = np.searchsorted(np.unique(y), y)

    def z(v):
        return (v[ids] - v[ids].mean()) / v[ids].std()

    cov = [] if extra is None else [extra[ids] - extra[ids].mean()]
    r_hep = cv_mcfadden(np.column_stack([z(hep)] + cov), y, K, ids)
    r_pci = cv_mcfadden(np.column_stack([z(pci)] + cov), y, K, ids)
    r_joint = cv_mcfadden(np.column_stack([z(hep), z(pci)] + cov), y, K, ids)
    return r_hep, r_pci, r_joint, r_joint - max(r_hep, r_pci)


THR_A = PRED["Pred 7.A"]["confirming_thresholds"]
out_7a = {}
for label, mask, extra in [("four_group", np.ones(len(group), bool), None), ("patient_only", patients, tbi)]:
    idx = np.flatnonzero(mask)
    r_hep, r_pci, r_joint, d = delta_r2_7a(idx, extra)
    br = np.random.default_rng(21)
    boot_d = [delta_r2_7a(br.choice(idx, len(idx)), extra)[3] for _ in range(40)]
    lo = np.percentile(boot_d, 2.5)
    out_7a[label] = dict(r=(r_hep, r_pci, r_joint), d=d, lo=lo)
    print(f"{label:<13} CV McFadden R²: HEP {r_hep:.3f}  PCI {r_pci:.3f}  joint {r_joint:.3f}  "
          f"ΔR² = {d:+.3f} (approx. 95% lower bound {lo:+.3f})")
ok_7a = all(v["d"] >= THR_A["delta_r_squared_minimum"] and v["lo"] > 0 for v in out_7a.values())
print(f"Pred 7.A (estimation-first; no framework vote) → {verdict(ok_7a)}")

## 4 — Pred 7.B: group discrimination and DMN double dissociation

In [ ]:
h_mcs, h_vs = hep[group == "MCS"], hep[group == "VS/UWS"]
t_b, p_b = ttest_ind(h_mcs, h_vs, alternative="greater")
d_b = (h_mcs.mean() - h_vs.mean()) / np.sqrt((h_mcs.var(ddof=1) + h_vs.var(ddof=1)) / 2)
means = [hep[group == g].mean() for g in GROUPS[:3]]
monotonic = all(a <= b for a, b in zip(means, means[1:]))
def centerd(v):
    return v - np.array([v[group == g].mean() for g in group])

r_dmn_pci, r_dmn_hep = pearsonr(dmn, pci)[0], pearsonr(dmn, hep)[0]
rc_dmn_pci, rc_dmn_hep = pearsonr(centerd(dmn), centerd(pci))[0], pearsonr(centerd(dmn), centerd(hep))[0]
confirmed_7B = p_b < 0.05 and d_b > 0.5 and monotonic and rc_dmn_pci > 0.50 and rc_dmn_hep < 0.20
print(f"HEP MCS > VS/UWS: d = {d_b:.2f}, p = {p_b:.1e}; patient gradient monotonic = {monotonic}")
print(f"Pooled:        DMN–PCI r = {r_dmn_pci:.2f}, DMN–HEP r = {r_dmn_hep:.2f}")
print(f"Group-centerd: DMN–PCI r = {rc_dmn_pci:.2f} (> 0.50), DMN–HEP r = {rc_dmn_hep:.2f} (< 0.20)")
print(f"Pred 7.B (secondary, BY-FDR q = 0.05 within protocol) → {verdict(confirmed_7B)}")

> **Specification flag.** If HEP and PCI both show the four-group gradient that Pred 7.B itself requires,
> any DMN measure that tracks severity also correlates with HEP across the pooled cohort. A pooled
> DMN–HEP r < 0.20 then conflicts with the gradient conjunct. This demo evaluates the dissociation on
> group-centerd (within-group) associations. The JSON leaves the negligible-association interval to be fixed
> before lock, and the manifest should also state the population and centring.

## 5 — Pred 7.C: CCRC perturbation versus auditory and carrier-matched sham controls

In [ ]:
THR_C = PRED["Pred 7.C"]["confirming_thresholds"]
effects = {}
for g, gain in [("MCS", 0.14), ("VS/UWS", 0.01)]:
    base = pci[group == g]
    n_g = len(base)
    d_hep = rng.normal(0.10 if g == "MCS" else 0.0, 0.05, n_g)
    effects[g] = dict(
        ccrc=gain + 0.5 * d_hep + rng.normal(0, 0.03, n_g),
        auditory=rng.normal(0.03 if g == "MCS" else 0.0, 0.03, n_g),
        carrier=rng.normal(0.01, 0.03, n_g),
        d_hep=d_hep,
    )
mcs = effects["MCS"]
margin = THR_C["margin_over_each_control_percent_baseline_pci"] / 100
p_aud = ttest_rel(mcs["ccrc"] - margin, mcs["auditory"], alternative="greater").pvalue
p_car = ttest_rel(mcs["ccrc"] - margin, mcs["carrier"], alternative="greater").pvalue
r_c = pearsonr(mcs["ccrc"], mcs["d_hep"])[0]
vs_change = effects["VS/UWS"]["ccrc"].mean()
confirmed_7C = (mcs["ccrc"].mean() * 100 >= THR_C["pci_increase_mcs_percent_minimum"]
                and max(p_aud, p_car) < THR_C["alpha_each_one_sided"]
                and r_c > THR_C["hep_pci_change_r_strict_minimum"] and abs(vs_change) < 0.05)
print(f"MCS ΔPCI (relative): CCRC {mcs['ccrc'].mean():+.1%}, auditory {mcs['auditory'].mean():+.1%}, "
      f"carrier sham {mcs['carrier'].mean():+.1%}; VS/UWS CCRC {vs_change:+.1%}")
print(f"CCRC − control ≥ 5% margin: vs auditory p = {p_aud:.1e}, vs carrier p = {p_car:.1e}; r(ΔPCI, ΔHEP) = {r_c:.2f}")
print(f"Pred 7.C → {verdict(confirmed_7C)}  [{PRED['Pred 7.C']['evaluation_status']}]")

## 6 — Pred 7.D: between-participant prognosis at 3 and 6 months

In [ ]:
def ols_r2(y, *cols):
    X = np.column_stack([np.ones(len(y))] + list(cols))
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    return 1 - np.sum((y - X @ b) ** 2) / np.sum((y - y.mean()) ** 2)


m = patients
rho3 = spearmanr(hep[observed3], crs3[observed3])[0]
rho6 = spearmanr(hep[m], crs6[m])[0]
base_crs = np.array([CRS_R_BASELINE[g] for g in group])[m] + rng.normal(0, 1.0, m.sum())
r2_clin_hep = ols_r2(crs6[m], base_crs, age[m], tbi[m], hep[m])
r2_clin_pci = ols_r2(crs6[m], base_crs, age[m], tbi[m], pci[m])
r2_joint = ols_r2(crs6[m], base_crs, age[m], tbi[m], hep[m], pci[m])
dr2_6 = r2_joint - max(r2_clin_hep, r2_clin_pci)
THR_D = PRED["Pred 7.D"]["confirming_thresholds"]
sub_i = min(rho3, rho6) > THR_D["spearman_r_each_timepoint_strict_minimum"]
sub_ii = dr2_6 >= THR_D["six_month_delta_r_squared_minimum"]
print(f"Spearman r(baseline HEP, CRS-R): 3 months {rho3:.2f}, 6 months {rho6:.2f}  → subclaim (i) {'met' if sub_i else 'not met'}")
print(f"6-month ΔR² (joint over clinical + better single biomarker) = {dr2_6:.3f} → subclaim (ii) {'met' if sub_ii else 'not met'}")
print("   (ΔR² here is the weaker marker's unique contribution beyond severity and the stronger marker)")
print(f"Pred 7.D → {verdict(sub_i and sub_ii)}")

## 7 — Visualize

In [ ]:
colors = ["#d6604d", "#FFCC00", "#66CC99", "#2166ac"]
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

ax = axes[0]
for k, g in enumerate(GROUPS[:3]):
    sel = observed3 & (group == g)
    ax.scatter(hep[sel], crs3[sel], color=colors[k], s=22, alpha=0.8, label=g)
b = np.polyfit(hep[observed3], crs3[observed3], 1)
xs = np.linspace(hep[observed3].min(), hep[observed3].max(), 50)
ax.plot(xs, np.polyval(b, xs), "k-", lw=1.2)
ax.set_xlabel("Baseline HEP (a.u.)")
ax.set_ylabel("CRS-R total, 3 months")
ax.set_title(f"Pred 7.G — primary\nLRT p = {p_lrt:.1e}, ΔR² = {res['dr2']:.3f}", fontsize=10)
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
ax.hist(boot, bins=25, color="#9966FF", alpha=0.8, edgecolor="white")
ax.axvline(SESOI, ls="--", color="k", lw=1, label="SESOI ΔR² = 0.05")
ax.axvspan(*ci90, color="#2166ac", alpha=0.15, label="90% CI")
ax.set_xlabel("Bootstrap marginal ΔR² (whole-site)")
ax.set_title("Pred 7.G — ΔR² vs SESOI", fontsize=10)
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[2]
x = np.arange(len(GROUPS))
ax.bar(x - 0.2, [hep[group == g].mean() for g in GROUPS], width=0.4, color=colors, alpha=0.85, edgecolor="white",
       label="HEP")
ax.bar(x + 0.2, [pci[group == g].mean() / pci.max() * hep.max() for g in GROUPS], width=0.4, color=colors,
       alpha=0.45, edgecolor="k", hatch="//", label="PCI (scaled)")
ax.set_xticks(x, GROUPS)
ax.set_title("Pred 7.B — four-group gradient", fontsize=10)
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[3]
lab = ["CCRC", "Auditory", "Carrier\nsham"]
ax.bar(lab, [mcs["ccrc"].mean(), mcs["auditory"].mean(), mcs["carrier"].mean()],
       yerr=[v.std() / np.sqrt(len(v)) for v in (mcs["ccrc"], mcs["auditory"], mcs["carrier"])],
       color=["#2166ac", "#AAAAAA", "#CCCCCC"], alpha=0.85, edgecolor="white", capsize=4, width=0.5)
ax.axhline(0.10, ls="--", color="k", lw=1, label="10% criterion")
ax.set_ylabel("ΔPCI (relative) in MCS")
ax.set_title("Pred 7.C — perturbation (conditional)", fontsize=10)
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Protocol 7 — DoC biomarkers · spec v1.1.0 · synthetic N = 110 (80 patients + 30 controls)", fontsize=12)
plt.tight_layout()
plt.show()